# One local model, three Java APIs

Models framework adapters consume the same `TextGenerationModel` contract. This notebook uses a deterministic in-process implementation so the adapter behavior is visible without downloading weights.

In [1]:
import com.integrallis.models.api.BackendDiagnostics;
import com.integrallis.models.api.SamplingOptions;
import com.integrallis.models.api.TextGenerationModel;
import com.integrallis.models.api.TokenStream;
import com.integrallis.models.langchain4j.ModelsChatModel;
import com.integrallis.models.spring.ai.ModelsSpringAiChatModel;
import org.springframework.ai.chat.prompt.Prompt;

In [2]:
class NotebookGenerationModel implements TextGenerationModel {
    public String modelName() {
        return "NotebookModel";
    }

    public BackendDiagnostics diagnostics() {
        return BackendDiagnostics.unavailable("notebook");
    }

    public void generate(String prompt, SamplingOptions options, TokenStream stream) {
        stream.onToken("local ");
        stream.onToken("answer");
        stream.onComplete();
    }
}

## Plain Java, LangChain4j, and Spring AI

The adapters change the application-facing API, not the inference engine or diagnostics.

In [3]:
TextGenerationModel engine = new NotebookGenerationModel();
var options = SamplingOptions.builder().temperature(0.0f).maxTokens(16).build();

System.out.println("plain java: " + engine.generate("question", options));

var langchain4j = new ModelsChatModel(engine, options);
System.out.println("langchain4j: " + langchain4j.chat("question"));

var springAi = new ModelsSpringAiChatModel(engine, options);
System.out.println("spring ai: " + springAi.call("question"));

var springTokens = springAi.stream(new Prompt("question"))
    .map(response -> response.getResult().getOutput().getText())
    .collectList()
    .block();
System.out.println("spring stream: " + String.join("", springTokens));
System.out.println("diagnostics: " + langchain4j.diagnostics().backend());

plain java: local answer


langchain4j: local answer


spring ai: local answer


spring stream: local answer


diagnostics: notebook
